In [ ]:
import pandas as pd
import numpy as np

# 1. Load both documents
dict_df = pd.read_csv('retail-data-dictionary.csv')
raw_df = pd.read_csv('retail-orders-raw.csv')

print("✅ Files loaded successfully!")
print(f"Dictionary rules: {len(dict_df)}, Raw orders: {len(raw_df)}")
display(dict_df)

# 2. DATA PROFILING
print("\n--- NULL check ---")
print(raw_df.isnull().sum())

print("\n--- Duplicate order_id ---")
print(raw_df[raw_df.duplicated('order_id', keep=False)])

# 3. CLEANING as per dictionary
raw_df['customer_segment'] = raw_df['customer_segment'].str.strip().str.title()
raw_df['payment_status'] = raw_df['payment_status'].str.strip().str.title()
raw_df['qty_num'] = pd.to_numeric(raw_df['quantity'], errors='coerce')
raw_df['disc_num'] = pd.to_numeric(raw_df['discount_pct'], errors='coerce').fillna(0)
raw_df['order_date_parsed'] = pd.to_datetime(raw_df['order_date'], errors='coerce')

clean_df = raw_df[
    (raw_df['qty_num']>0) &
    (raw_df['disc_num'].between(0,100)) &
    (raw_df['city'].notna()) &
    (raw_df['order_date_parsed'].notna())
].drop_duplicates(subset=['order_id'])

print(f"\nCleaned rows: {len(clean_df)} (Original: {len(raw_df)})")

# 4. KPI CALCULATION
clean_df['revenue'] = clean_df['qty_num'] * clean_df['unit_price'] * (1 - clean_df['disc_num']/100)

total_revenue = clean_df[clean_df['payment_status']=='Paid']['revenue'].sum()
total_orders = clean_df['order_id'].nunique()
aov = total_revenue / total_orders

print(f"\n--- FINAL KPIs ---")
print(f"Total Revenue (Paid): {total_revenue:.2f} INR")
print(f"Total Orders: {total_orders}")
print(f"AOV: {aov:.2f} INR")
print(f"Avg Discount: {clean_df['disc_num'].mean():.2f}%")
print("\nRevenue by Category:")
print(clean_df.groupby('category')['revenue'].sum())
print("\nOrders by City:")
print(clean_df.groupby('city')['order_id'].count())

# 5. Save files for submission
clean_df.to_csv('retail-orders-cleaned.csv', index=False)
print("\n✅ File saved: retail-orders-cleaned.csv")

✅ Files loaded successfully!
Dictionary rules: 9, Raw orders: 12


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...



--- NULL check ---
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64

--- Duplicate order_id ---
  order_id  order_date customer_segment       city      category quantity  \
3  RT-1004  2026-01-07     Professional  Hyderabad  Learning Kit        3   
4  RT-1004  2026-01-07     Professional  Hyderabad  Learning Kit        3   

   unit_price  discount_pct payment_status  
3         799           5.0           Paid  
4         799           5.0           Paid  

Cleaned rows: 5 (Original: 12)

--- FINAL KPIs ---
Total Revenue (Paid): 4514.35 INR
Total Orders: 5
AOV: 902.87 INR
Avg Discount: 6.00%

Revenue by Category:
category
Course Access    4047.30
Learning Kit     4514.35
Name: revenue, dtype: float64

Orders by City:
city
Chennai      3
Delhi        1
Hyderabad    1
Name: order_id, dtype: int64

✅ File saved: retail-orders-